# Source Ledger — quickstart

**Run All.** Nothing to edit, nothing to install by hand, no API keys, and no
terminal at any point.

This notebook does one full pass of what the accelerator is for: it checks what
this network can reach, asks a question, and keeps the answer *and the way the
answer was found* as a governed row — which it then reads back, queries in SQL,
and writes out as the appendix a reviewer files.

About a minute end to end; the preflight is the slow part. It writes two things,
both gitignored and both under `data/`: the record itself at `data/source_ledger.db`,
and a CSV of it at `data/exports/review-appendix.csv`. Anything else it needs, it
installs into this kernel as it goes.

In [ ]:
# --- bootstrap ---
# A notebook has no __file__, so the working directory is the only anchor there is.
# It is usually the notebook's own directory — but VS Code's jupyter.notebookFileRoot
# can point a kernel at the workspace folder instead, and `jupyter lab` started from
# elsewhere inherits wherever it was started. So search upward for the repository
# rather than assuming, and say so plainly when it is not there.
import sys
from pathlib import Path


def find_root(start=None):
    """The Source Ledger repository at or above `start`."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        # Two markers, not one: scripts/new-accelerator.sh copies METADATA.yaml into
        # a fresh accelerator, so on its own it would match the wrong repository.
        if (candidate / "METADATA.yaml").is_file() and (candidate / "data" / "db.py").is_file():
            return candidate
    raise RuntimeError(
        f"No Source Ledger repository at or above {here}. "
        "Open quickstart.ipynb from inside the cloned repository.")


ROOT = find_root()
# The layers are directories, not installed packages, so they go on the path the
# same way app/server.py puts them there at runtime. ROOT itself is included for
# the same reason tests/conftest.py adds it: `app` is a package, reached from the
# directory above it.
for layer in ("retrieval", "data", "scripts", "."):
    path = str((ROOT / layer).resolve())
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"repository  {ROOT}")
print(f"kernel      {sys.executable}")

## 1. Dependencies

The search library is one package. If this kernel does not have it, this cell
installs it — into *this* interpreter, which is frequently not the `pip` on your
PATH, and is the single most common reason a "but I installed it" notebook fails.

If the install cannot be done for you, nothing raises: the cell says what to run,
and every cell below reports itself skipped instead of throwing a traceback.

In [ ]:
import importlib
import importlib.util
import subprocess


def ensure(modules, requirements, what):
    """Make sure `modules` are importable here, installing `requirements` if not.

    Returns True when they are usable. Never raises: a quickstart that dies on its
    dependency cell has failed at the one job it has.
    """
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if not missing:
        return True

    print(f"Installing {what} ({', '.join(missing)} missing)...")
    install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
    # A managed runtime image — a Cloudera AI session, say — usually has a
    # site-packages the session user cannot write to, and there the install has to
    # go to the user site instead. Not offered inside a virtualenv, where pip
    # rejects --user outright because the user site is not on its path.
    attempts = [install] if sys.prefix != sys.base_prefix else [install, install + ["--user"]]
    for attempt in attempts:
        try:
            subprocess.check_call(attempt)
            break
        except (subprocess.CalledProcessError, OSError) as exc:
            failure = exc
    else:
        print(f"  Could not install it automatically ({failure}). Run this yourself:\n")
        print(f"    {sys.executable} -m pip install -r {requirements}\n")
        print("  Then restart the kernel and Run All again.")
        return False

    # A package installed after the interpreter started is invisible until the
    # import system is told to look again.
    importlib.invalidate_caches()
    still_missing = [m for m in missing if importlib.util.find_spec(m) is None]
    if still_missing:
        print(f"  Installed, but {', '.join(still_missing)} is still not importable.")
        print("  Restart the kernel and Run All again.")
        return False
    print("  Installed.")
    return True


READY = ensure(["ddgs"], ROOT / "retrieval" / "requirements.txt", "the search library")
if READY:
    print("ready")

## 2. What can this network actually reach?

The same preflight `make doctor` runs, probing each corpus and **each web engine
individually** — a single blocked engine is worth seeing by name rather than
hidden inside the pool.

This is the slow cell: 10–40 seconds, longer if something is timing out. A blocked
engine here is a measurement of this network, not a defect. The next cell picks a
provider from whatever answered, so a locked-down network gets a readable
diagnosis instead of a failed search three cells later.

In [ ]:
if READY:
    import doctor

    probes = doctor.rows()
    for label, status, detail, seconds in probes:
        print(f"  {doctor.MARK[status]}  {label:16} {seconds:5.1f}s  {detail}")

    healthy = [label for label, status, _, _ in probes if status == doctor.OK]
    ENGINES = [label.removeprefix("web: ") for label in healthy if label.startswith("web: ")]
    APIS = [label for label in healthy if not label.startswith("web: ")]

    # Prefer the open web when any engine answered; otherwise search a corpus this
    # network can actually reach, rather than demonstrating a failure.
    PROVIDER = "ddgs" if ENGINES else (APIS[0] if APIS else None)
    if PROVIDER is None:
        READY = False
        print("\n  Nothing is reachable from here. Check the network, a proxy, or a")
        print("  TLS-intercepting firewall; the rows above name which it is.")
    else:
        print(f"\n  Searching with: {PROVIDER}"
              + (f" via {', '.join(ENGINES)}" if ENGINES else ""))
else:
    print("skipped - see the dependency cell above")

## 3. One question, recorded

Edit `QUERY` and re-run this cell as often as you like; every run adds a row.

The part worth watching is what gets *stored*. Options this corpus does not apply
are recorded as `NULL`, not as the value passed in — so the record can never claim
a filter that never ran. That single rule is what makes the table defensible months
later, and it is enforced in one place, `data/record.py`, for the dashboard, the
terminal, and this notebook alike.

In [ ]:
QUERY = "How people are using Cloudera Agentic Studio?"   # <- edit me

if READY:
    import db
    import record
    import source_ledger

    # The store has to exist before the first row, exactly as app/server.py
    # ensures at import. Idempotent, and it migrates an older store in place.
    db.init_db()

    try:
        result = record.run(QUERY, provider=PROVIDER, max_results=10, backend=ENGINES)
    except source_ledger.EngineError as exc:
        # Every engine failed and each said why. The network can go down between
        # the preflight and this cell, so this is not belt-and-braces.
        print(f"{record.label(PROVIDER)} search failed - no engine answered:")
        for engine, reason in exc.failures:
            print(f"  {engine}: {reason}")
        result = None
    except Exception as exc:
        print(f"{record.label(PROVIDER)} search failed - {exc}")
        result = None

    if result and result["urls"]:
        print(f'{len(result["urls"])} urls, recorded as search #{result["search_id"]}:\n')
        for url in result["urls"]:
            print(f"  {url}")
    elif result:
        # Not an error: the corpus answered and had nothing.
        print(f'No results. {record.label(PROVIDER)} had nothing for "{QUERY}".')
        print("Try editing QUERY above - a scholarly corpus will not match a")
        print("navigational web query, and vice versa.")
else:
    print("skipped - see the cells above")

## 4. The record

The deliverable. Every row carries the question and the options that produced it,
so the search can be reproduced or audited by someone who was not here.

Read the option columns carefully — they say two different things:

* **`n/a`** — this corpus has no such option. Nothing was filtered.
* **`any`** — it has one, and this search chose not to use it.

Collapsing those two is exactly the overstatement the record exists to prevent,
which is also why this table is hand-rendered rather than handed to pandas: a
DataFrame prints both as a blank cell. The rendering lives in
[`data/present.py`](data/present.py) rather than in this cell, so the rule is
tested once and holds for the notebook, the dashboard, and the export alike.


In [ ]:
from IPython.display import HTML, display

import present

# Deliberately ungated: reading the record back needs no search library and no
# network, which is the point of keeping it in a file rather than behind a
# service. A reader with neither still gets the table.
if present.store() is None:
    print(f"No store at {present.db.DB_PATH} yet - run section 3 first.")
else:
    display(HTML(present.searches(limit=10)))
    print("n/a = this corpus has no such option.  any = it has one, unused.")
    # If you would rather have a DataFrame, and can live with it blurring the two:
    # import pandas as pd; pd.DataFrame(present.db.list_searches(limit=10))


## 5. The record, in SQL

`data/source_ledger.db` is a SQLite file, so the most direct way to interrogate the
record is the one an auditor would reach for anyway — no service to start, no port
to reach, nothing between the question and the row.

Edit `SQL` and re-run as often as you like. The connection is opened **read-only**,
so nothing typed into this cell can alter the record it exists to examine.

Two tables, and [`data/schema.sql`](data/schema.sql) is the whole of it:

* `searches` — `id`, `query`, `created_at`, `provider`, `region`, `safesearch`,
  `timelimit`, `backend`, `max_results`
* `search_urls` — `id`, `search_id`, `position`, `url`, `provider`

Keep section 4's `NULL` / `''` rule in mind while writing against this, because it
is where the distinction earns its keep: `WHERE timelimit IS NULL` asks *which
searches ran on a corpus that has no time filter*, and `WHERE timelimit = ''` asks
*which had one and chose not to use it*. Different questions, and only one of them
is about the corpus.

In [ ]:
import html
import sqlite3
from contextlib import closing

from IPython.display import HTML, display

import db

SQL = """
SELECT s.id,
       s.created_at,
       s.query,
       s.provider,
       s.timelimit,
       s.backend,
       COUNT(u.id) AS urls
FROM searches AS s
LEFT JOIN search_urls AS u ON u.search_id = s.id
GROUP BY s.id
ORDER BY s.id DESC
LIMIT 10
"""   # <- edit me


def ask(sql, parameters=()):
    """Run `sql` against the record; return its column names and its rows.

    Opened read-only through a file: URI, not by convention — this cell exists to
    interrogate a governed store, and an UPDATE typed into an audit query is the
    one accident it must not be able to have. SQLite raises instead of writing.

    closing(), because sqlite3's own context manager ends the transaction but
    leaves the handle open — the same reason data/db.py wraps its connections.
    """
    uri = f"{db.DB_PATH.resolve().as_uri()}?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as conn:
        cursor = conn.execute(sql, parameters)
        return [column[0] for column in cursor.description], cursor.fetchall()


def show(columns, rows):
    """The result set, with NULL and unset kept apart.

    A plain table prints both as an empty cell, which is the single confusion this
    record is built to prevent — so the distinction survives into SQL output too.
    """
    def cell(value):
        if value is None:
            return '<span style="opacity:.5">n/a</span>'
        if value == "":
            return '<span style="opacity:.5">any</span>'
        return html.escape(str(value))

    head = "".join(f'<th style="text-align:left;padding:4px 10px">{html.escape(c)}</th>'
                   for c in columns)
    body = "".join(
        '<tr style="border-top:1px solid rgba(128,128,128,.35)">'
        + "".join(f'<td style="padding:4px 10px;vertical-align:top;max-width:28em">'
                  f'{cell(value)}</td>' for value in row)
        + "</tr>" for row in rows)
    return (f'<table style="border-collapse:collapse;font-size:.9em">'
            f'<tr style="border-bottom:2px solid rgba(128,128,128,.6)">{head}</tr>'
            f'{body}</table>')


# Not gated on READY: reading the record needs no search library and no network,
# which is the whole point of keeping it in a file rather than in a service.
if not db.DB_PATH.exists():
    print(f"No store at {db.DB_PATH} yet - run section 3 first.")
else:
    try:
        columns, rows = ask(SQL)
    except sqlite3.Error as exc:
        # A typo in SQL is the expected failure here, and sqlite3 says where.
        print(f"SQL error: {exc}")
    else:
        display(HTML(show(columns, rows)))
        print(f"{len(rows)} row{'' if len(rows) == 1 else 's'} from {db.DB_PATH}")
        print("n/a = this corpus has no such option.  any = it has one, unused.")

## 6. Export it

One row per URL, with the provenance of its search denormalized onto it — the shape
a reviewer sorts by domain and filters by provider, and the same shape the lakehouse
curates into. `NULL` is spelled out because CSV has only one kind of empty cell and
this record needs two.

This cell writes the file and then shows you the top of it. It lands in
`data/exports/` — beside the store it was drawn from rather than in whatever
directory your kernel happens to have started in, and in its own folder because
`data/` is the Ingest layer's source and a deliverable is not source. Gitignored,
for exactly the reason the store is: every row carries the query text verbatim.

The flattening is [`data/present.py`](data/present.py)'s, not this cell's, and
`scripts/cli.py export` runs the same code — so the file a reviewer gets is
byte-identical whichever of the two produced it.

Re-running overwrites it, so the file is always the record as it stands now.


In [ ]:
import present

# present.export() picks the path: data/exports/review-appendix.csv, beside the
# store it is drawn from but in its own directory, because data/ is the Ingest
# layer's source and a deliverable is not source. Not the working directory
# either - a notebook's cwd depends on how the kernel was started, and an export
# nobody can find is an export that did not happen.
if present.store() is None:
    print(f"No store at {present.db.DB_PATH} yet - run section 3 first.")
else:
    PATH, ROWS = present.export()

    print(f"Wrote {len(ROWS)} url rows to {PATH.relative_to(ROOT).as_posix()}")
    if not ROWS:
        print("The record is empty, so that is the header alone - run section 3 first.")
    print()

    for line in present.head(PATH):
        print(line)

    print()
    print("The same rows from a terminal, wherever you point it:")
    print("    python scripts/cli.py export --format csv --out review-appendix.csv")
